In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# we will only import certain module from those libraries
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
#from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.covariance import EllipticEnvelope
from sklearn.ensemble import IsolationForest
from random import randrange
from datetime import datetime
import math
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
# the file path is the archive location of the file in our computer
file_path = './data/raw_data/train/1000_chg.csv'
data = pd.read_csv(file_path) # using pandas library (pd) to read the csv file.
# After reading the file, it will be used as a Pandas DataFrame.
# Pandas DataFrame is a special data structure from Pandas Library that is two-dimensional,
# size-mutable, potentially heterogeneous tabular data.

In [3]:
data.head() #데이터 초반 5개 보여줌

,Date,Time,SerialNumber,Voltage,Current,RSOCmin,RSOCmax,RSOCavg,USOCmin,USOCmax,...,M12T01,M12T02,M13T01,M13T02,M14T01,M14T02,M15T01,M15T02,M16T01,M16T02
0,2020-08-04,15:51:49,1000,641.3,0.0,33.43,34.29,33.84,33,34,...,31.6,31.7,31.8,31.5,31.7,31.8,31.8,32.0,31.6,31.7
1,2020-08-04,15:51:50,1000,641.3,0.0,33.43,34.29,33.84,33,34,...,31.6,31.7,31.8,31.5,31.7,31.9,31.8,32.0,31.6,31.7
2,2020-08-04,15:51:51,1000,641.3,0.0,33.43,34.29,33.84,33,34,...,31.6,31.7,31.8,31.5,31.7,31.9,31.8,32.0,31.6,31.7
3,2020-08-04,15:51:52,1000,641.3,0.0,33.43,34.29,33.84,33,34,...,31.6,31.7,31.8,31.5,31.7,31.8,31.8,32.0,31.6,31.7
4,2020-08-04,15:51:53,1000,641.3,0.0,33.43,34.29,33.84,33,34,...,31.6,31.7,31.8,31.5,31.7,31.8,31.8,32.0,31.6,31.7


In [4]:
# There are 231 columns, and some of the names are shown in below.
data.columns

Index(['Date', 'Time', 'SerialNumber', 'Voltage', 'Current', 'RSOCmin',
       'RSOCmax', 'RSOCavg', 'USOCmin', 'USOCmax',
       ...
       'M12T01', 'M12T02', 'M13T01', 'M13T02', 'M14T01', 'M14T02', 'M15T01',
       'M15T02', 'M16T01', 'M16T02'],
      dtype='object', length=231)

In [5]:
 # info() 메소드는 Data frame의 요약을 보여준다. 
 # 표시되는 정보는 데이터 수가 6009개이며, 231개의 속성이 있음을 나타낸다. 
 # 또한, 데이터의 유형, non-null value 및 메모리 사용 등의 정보가 표시된다.#
data.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6009 entries, 0 to 6008
Columns: 231 entries, Date to M16T02
dtypes: float64(219), int64(10), object(2)
memory usage: 10.6+ MB


In [6]:
data.shape #- 데이터의 형태를 (rows, columns) 형식으로 보여준다. 

(6009, 231)

In [7]:
data.dtypes #각 속성의 데이터 유형을 표시하기 위해서는 다음 구문을 사용한다.

Date             object
Time             object
SerialNumber      int64
Voltage         float64
Current         float64
                 ...   
M14T02          float64
M15T01          float64
M15T02          float64
M16T01          float64
M16T02          float64
Length: 231, dtype: object

In [8]:
data.nunique(axis = 0) #각 속성의 고유한 값 (unique value)의 수를 표시하기 위해서는 다음 구문을 사용한다.

Date               1
Time            6009
SerialNumber       1
Voltage          712
Current          286
                ... 
M14T02            46
M15T01            50
M15T02            43
M16T01            48
M16T02            40
Length: 231, dtype: int64

In [9]:
data[data['Time'].isin(data['Time'][data['Time'].duplicated()])].sort_values("Time") #중복된 행이 있는지 보여줌, 0행이면 없다는 의미

,Date,Time,SerialNumber,Voltage,Current,RSOCmin,RSOCmax,RSOCavg,USOCmin,USOCmax,...,M12T01,M12T02,M13T01,M13T02,M14T01,M14T02,M15T01,M15T02,M16T01,M16T02


In [10]:
#describe() 메소드는 데이터의 기초 통계를 요약하여 보여준다. 
#데이터의 각 속성에대하여 데이터수(count), 평균 (mean), 표준편차 (std), 최소값 (min), 25/50/75 분위수, 그리고 최대값(max) 등의 값을 표시한다.
data.describe() 

,SerialNumber,Voltage,Current,RSOCmin,RSOCmax,RSOCavg,USOCmin,USOCmax,USOCavg,SOH,...,M12T01,M12T02,M13T01,M13T02,M14T01,M14T02,M15T01,M15T02,M16T01,M16T02
count,6009.0,6009.000000,6009.000000,6009.000000,6009.000000,6009.000000,6009.000000,6009.000000,6009.000000,6009.0,...,6009.000000,6009.000000,6009.000000,6009.000000,6009.000000,6009.000000,6009.000000,6009.000000,6009.000000,6009.000000
mean,1000.0,690.951207,-35.552072,70.717998,71.577998,71.127998,77.457813,78.155933,77.799135,0.0,...,34.873506,35.171443,34.987552,34.584140,35.097703,34.887702,35.188883,35.008903,35.002247,34.491147
std,0.0,24.973541,25.325789,19.037510,19.037510,19.037510,22.550164,22.239537,22.393269,0.0,...,1.389282,1.448724,1.358591,1.333149,1.422706,1.326838,1.433096,1.294315,1.391183,1.215350
min,1000.0,641.300000,-57.300000,33.430000,34.290000,33.840000,33.000000,34.000000,34.000000,0.0,...,31.600000,31.700000,31.800000,31.500000,31.700000,31.800000,31.700000,32.000000,31.600000,31.700000
25%,1000.0,666.400000,-55.400000,54.090000,54.950000,54.500000,58.000000,59.000000,58.000000,0.0,...,34.100000,34.400000,34.200000,33.800000,34.300000,34.100000,34.400000,34.300000,34.300000,33.800000
50%,1000.0,698.100000,-54.100000,75.850000,76.710000,76.260000,83.000000,84.000000,84.000000,0.0,...,35.400000,35.700000,35.500000,35.100000,35.600000,35.400000,35.700000,35.500000,35.600000,35.000000
75%,1000.0,714.800000,0.000000,89.600000,90.460000,90.010000,100.000000,100.000000,100.000000,0.0,...,36.000000,36.400000,36.100000,35.700000,36.300000,36.000000,36.400000,36.100000,36.100000,35.500000
max,1000.0,719.000000,0.000000,89.600000,90.460000,90.010000,100.000000,100.000000,100.000000,0.0,...,36.200000,36.500000,36.300000,35.800000,36.400000,36.300000,36.600000,36.200000,36.300000,35.600000


In [11]:
# array_attributes = the names of the columns we would like the histograms for
# cols_number = indicate how many columns for the grid
def showHistograms(data, histogram_attributes, cols_number, height, width):

 rows_number = math.ceil(len(histogram_attributes) / cols_number)
 # rows number calculated automatically
 fig = make_subplots(rows = rows_number, cols = cols_number,
        subplot_titles = (histogram_attributes))

 for i in range(len(histogram_attributes)):
    legend = histogram_attributes[i] + " count"
 # cols and rows start in 1, not in 0, thus we add one.
    row_number = i % cols_number + 1
    col_number = math.ceil((i+1) / cols_number)
    fig.append_trace(go.Histogram(name = legend, x = data[histogram_attributes[i]]),
        row_number, col_number)
 fig.update_layout(height = height, width = width,
    title_text = "Histograms of Selected Attributes")
 fig.show()

In [12]:
histogram_attributes = ["Voltage", "Current", "RSOCavg", "USOCavg", "Power", "M16T02",
	 	 	 "Vmax", "Tavg"]
# attributes we want in our histogram grid
showHistograms(data, histogram_attributes, 3, 900, 900)

In [13]:
# 히트맵 그리는 함수. 상관계수 수치가 1에 가까울수록 강한 양의 상관관계, -1에 가까울수록 강한 음의 상관관계를 나타냄.
# Procedure for Drawing Correlation Heatmap with Coefficient Text Printed
# Input parameter: the dataframe name, the width for heatmap graph, and the number of column
# to visualize.
# Output: correaltion heatmap of the input dataframe with size (width X width)
def plotCorrelationMatrixText(df, graphSize, n):
    df = df[[col for col in df if df[col].nunique() >1]] # keep columns where there are
 # more than 1 unique values
    df = df.iloc[:, :n+1] #get the first n columns
    if df.shape[1] <2: #check if there are more than one column in dataset.
 #If only one column in a dataset, print error message.
        print(f'No correlation plots shown: ')
        print(f'The number of non-NaN or constant columns ({df.shape[1]}) is less than 2')
        return
    corr = df.corr() #calculate pearson correlation between all attributes in dataset

    heatmap = go.Heatmap(
    z=corr,
    x=corr.columns,
    y=corr.columns,
    text=corr,
    texttemplate="%{text:.2f}",
    textfont={"size":7}

    )

    layout = go.Layout(
        title_text ="Correlation Matrix",
        width=graphSize,
        height=graphSize,
    )
    fig = go.Figure(data=[heatmap], layout=layout)
    fig.show()

In [14]:
plotCorrelationMatrixText(data, 800, 35) #create heatmap correlation for dataframe data2
 #with 10 pixels width and for 35 first column

In [15]:
# 직접 짤 수 있어야함. 추후 공부해보자.
# 이 산점도 행렬은 상관계수 수치만으로는 알 수 없는 비선형적 관계를 확인하는 방법임.
# 완벽한 직선은 오히려 같은 수치를 다른 도메인으로 관찰한 것으로 판단할 수 있고 일정한 방향성을 갖고 
# Procedure for Drawing Scatter Plot Matrix
# Input parameter: the dataframe name, the number of column to visualize, the size of graph,
# the text size of graph
# Output: Scatter matrix graph with specified size
def plotScatterMatrix(df, n, plotSize):
    df = df.select_dtypes(include = [np.number]) # keep only numerical columns
    df = df.dropna('columns') #drop column with missing data
    df = df[[col for col in df if df[col].nunique() > 1]]
    # keep columns where there are more than 1 unique values
    df = df.iloc[:, :n] #get the first n columns
    columnNames = list(df) #list of all attributes
    df = df[columnNames]
    layout = go.Layout(
        title_text = "Scatter matrix",
        width = plotSize,
        height = plotSize,)
    fig = px.scatter_matrix(df)
    fig.update_layout(
        title = '<span style="font-size: 16px;">Scatter and Density Plot</span>',
        width = plotSize, height=plotSize,
        font = dict(size = 8))
    fig.show()

In [16]:
plotScatterMatrix(data, 10, 800) #draw scatter matrix for dataframe dataset2 with size 800px

In [17]:
data.isna().sum()
#속성별 null 값의 수를 확인할 수 있음. 0이면 결측치가 없다는 의미

Date            0
Time            0
SerialNumber    0
Voltage         0
Current         0
               ..
M14T02          0
M15T01          0
M15T02          0
M16T01          0
M16T02          0
Length: 231, dtype: int64

In [18]:
null_data = data[data.isnull().any(axis=1)]
null_data
#null_data[null_data.columns[null_data.isnull().any()]]
#속성 별 결측치가 있는 행을 확인할 수 있음. 0행이면 결측치가 없다는 의미

,Date,Time,SerialNumber,Voltage,Current,RSOCmin,RSOCmax,RSOCavg,USOCmin,USOCmax,...,M12T01,M12T02,M13T01,M13T02,M14T01,M14T02,M15T01,M15T02,M16T01,M16T02


In [19]:
def removeConstant(df, n):
 df = df[[col for col in df if df[col].nunique() > n]]
 return df
data1 = removeConstant(data, 1)
removed_cols = set(data.columns) - set(data1.columns)
print(f"제거된 컬럼 이름: {sorted(removed_cols)}")
#칼럼 제거 후 데이터 이름 확인
#수치가 변하지 않고 하나인 칼럼을 제거함. 

제거된 컬럼 이름: ['Date', 'DchgImax', 'SOH', 'SerialNumber']


In [20]:
print("\n--- 제거된 컬럼별 상세 정보 ---") #제거된 칼럼의 정보 출력하는 블록
for col in sorted(removed_cols):
    # nunique(): 해당 컬럼의 고유값 개수 (제거 대상이므로 보통 1이 나와야 정상)
    n_unique = data[col].nunique()

    # unique(): 실제로 어떤 값(들)으로 채워져 있었는지 배열로 반환.
    # 상수 컬럼이면 원소가 1개짜리 배열이 나온다. 예: array([0.])
    unique_val = data[col].unique()

    # isnull().mean(): 결측 비율. 혹시 "값이 거의 다 NaN이고 딱 1개 값만 있는" 경우인지
    # 구분해서 보기 위함 -> 센서 고장(항상 같은 값) vs 데이터 누락(대부분 NaN)을 구별하는 단서
    missing_ratio = data[col].isnull().mean()

    print(f"{col:15s} | 고유값 개수: {n_unique} | 값: {unique_val} | 결측비율: {missing_ratio:.1%}")


--- 제거된 컬럼별 상세 정보 ---
Date            | 고유값 개수: 1 | 값: ['2020-08-04'] | 결측비율: 0.0%
DchgImax        | 고유값 개수: 1 | 값: [200] | 결측비율: 0.0%
SOH             | 고유값 개수: 1 | 값: [0] | 결측비율: 0.0%
SerialNumber    | 고유값 개수: 1 | 값: [1000] | 결측비율: 0.0%


In [21]:
data.shape
data1.shape

(6009, 227)

In [22]:
def identify_outliers(df, c):
    #set the constant for IQR boundary
    constant = float(c)
    # calculate Q1 and Q3
    Q1 = df.quantile(0.25)
    Q3 = df.quantile(0.75)
    # calculate the IQR
    IQR = Q3 - Q1
    # filter the dataset with the IQR
    IQR_outliers = df[((df.lt(Q1 - constant * IQR)) | (df.gt(Q3 + constant * IQR))).any(axis=1)]
    IQR_outliers = pd.DataFrame(IQR_outliers)
    return IQR_outliers

def remove_outliers(df, c):
    #find outliers
    df = pd.DataFrame(df)
    outliers = identify_outliers(df, c)
    #remove outliers
    df_out = pd.DataFrame(outliers)
    df.drop(df_out.index, inplace = True)
    return df

In [23]:
def handleMissingValue(data):
    df = data.copy()
    numeric_df = df.columns[df.dtypes != 'object']
    categorical_df = df.columns[df.dtypes == 'object']
    #handling missing value for categorical
    for i in categorical_df:
        df[i].fillna(df[i].mode()[0], inplace = True)

    df_flag_null = df.isnull()
    i, c = np.where(df_flag_null)
    #handling missing value for numerical
    for j in range(len(i)):
    #if missing value in the beginning
        if i[j] == 0:
            s = df.iloc[:, c[j]]
            id_s = s.notna().idxmax()
            val = df.iat[id_s, c[j]]
            df.iat[i[j], c[j]] = val
            #df.iloc[[i[j]], [c[j]]] = df.iloc[[id_s], [c[j]]]
            #print(type(val))
        #if missing value in the end
        elif i[j] == (len(df) - 1):
            s = df.iloc[:, c[j]]
            id_s = s.notna()[::-1].idxmax()
            val = df.iat[id_s, c[j]]
            df.iat[i[j], c[j]] = val
        #if missing value in the middle
        else:
            low = df.iat[i[j] - 1, c[j]]
            high = df.iat[i[j] + 1, c[j]]
            print(high)
            if(math.isnan(high)):
                val = low
            else:
                val = (low+high) / 2
                df.iat[i[j], c[j]] = val

    return df

In [24]:
data2 =handleMissingValue(data1)

data_numeric = data2.copy() #복사본 만듦
#그 데이터에서 IQR을 이용하여 이상치를 찾아냄. 4.0은 상수값으로, IQR의 몇 배를 기준으로 이상치를 판단할지 정함.
#IQR은 75-25% 데이터의 범위로 75%라인에서 4*IQR 만큼 벗어난 값을 이상치로 보겠다는 이야기 Q3 + 4*IQR, Q1 - 4*IQR이 이상치 기준선
#아래 함수는 이상치가 있는 행을 찾아서 반환함. 이상치가 있는 행은 제거해야함.
a = identify_outliers(data_numeric[:], 4.0) 
print(len(a))

a.index
#근데 이상치가 없을거임

0


Int64Index([], dtype='int64')

In [25]:
#단순 컬럼 이름 확인, 다음 단계인 전압, 온도 데이터만 추출하는 과정에서 컬럼 이름을 확인하기 위해 작성한 코드임.
columns_list2 = data2.columns
for i in range(len(columns_list2)):
 print(i, columns_list2[i])

0 Time
1 Voltage
2 Current
3 RSOCmin
4 RSOCmax
5 RSOCavg
6 USOCmin
7 USOCmax
8 USOCavg
9 Power
10 ChgPmax
11 DchgPmax
12 ChgImax
13 Vmin
14 Vmax
15 DV
16 Tmin
17 Tmax
18 Tavg
19 M01CV01
20 M01CV02
21 M01CV03
22 M01CV04
23 M01CV05
24 M01CV06
25 M01CV07
26 M01CV08
27 M01CV09
28 M01CV10
29 M01CV11
30 M02CV01
31 M02CV02
32 M02CV03
33 M02CV04
34 M02CV05
35 M02CV06
36 M02CV07
37 M02CV08
38 M02CV09
39 M02CV10
40 M02CV11
41 M03CV01
42 M03CV02
43 M03CV03
44 M03CV04
45 M03CV05
46 M03CV06
47 M03CV07
48 M03CV08
49 M03CV09
50 M03CV10
51 M03CV11
52 M04CV01
53 M04CV02
54 M04CV03
55 M04CV04
56 M04CV05
57 M04CV06
58 M04CV07
59 M04CV08
60 M04CV09
61 M04CV10
62 M04CV11
63 M05CV01
64 M05CV02
65 M05CV03
66 M05CV04
67 M05CV05
68 M05CV06
69 M05CV07
70 M05CV08
71 M05CV09
72 M05CV10
73 M05CV11
74 M06CV01
75 M06CV02
76 M06CV03
77 M06CV04
78 M06CV05
79 M06CV06
80 M06CV07
81 M06CV08
82 M06CV09
83 M06CV10
84 M06CV11
85 M07CV01
86 M07CV02
87 M07CV03
88 M07CV04
89 M07CV05
90 M07CV06
91 M07CV07
92 M07CV08
93 M07CV09


In [26]:
#셀 전압 온도 데이터만을 추출해서 새로 데이터프레임을 만든다
df_train_0 = data2
columns_vt = columns_list2[18:226]
print(columns_vt)
df_train = df_train_0[columns_vt]
df_train.head(5)

Index(['Tavg', 'M01CV01', 'M01CV02', 'M01CV03', 'M01CV04', 'M01CV05',
       'M01CV06', 'M01CV07', 'M01CV08', 'M01CV09',
       ...
       'M11T02', 'M12T01', 'M12T02', 'M13T01', 'M13T02', 'M14T01', 'M14T02',
       'M15T01', 'M15T02', 'M16T01'],
      dtype='object', length=208)


,Tavg,M01CV01,M01CV02,M01CV03,M01CV04,M01CV05,M01CV06,M01CV07,M01CV08,M01CV09,...,M11T02,M12T01,M12T02,M13T01,M13T02,M14T01,M14T02,M15T01,M15T02,M16T01
0,31,3.646,3.645,3.645,3.645,3.645,3.644,3.645,3.645,3.645,...,31.7,31.6,31.7,31.8,31.5,31.7,31.8,31.8,32.0,31.6
1,31,3.646,3.645,3.645,3.645,3.645,3.644,3.645,3.645,3.645,...,31.7,31.6,31.7,31.8,31.5,31.7,31.9,31.8,32.0,31.6
2,31,3.646,3.645,3.645,3.645,3.645,3.644,3.645,3.645,3.645,...,31.7,31.6,31.7,31.8,31.5,31.7,31.9,31.8,32.0,31.6
3,31,3.646,3.645,3.645,3.645,3.645,3.644,3.645,3.645,3.645,...,31.7,31.6,31.7,31.8,31.5,31.7,31.8,31.8,32.0,31.6
4,31,3.646,3.645,3.645,3.645,3.645,3.644,3.645,3.645,3.645,...,31.7,31.6,31.7,31.8,31.5,31.7,31.8,31.8,32.0,31.6


In [27]:
# Save the processed (train) data for later
df_train.to_csv('./data/preprocessed/train/1000_chg.csv', index = False)

In [28]:
#테스트 데이터도 동일하게 전처리 진행
# Next, 용량 불량 테스트 데이타
# the file path is the archive location of the file in our computer
file_path = './data/raw_data/test/Test07_NG_dchg.csv'
df_test2 = pd.read_csv(file_path) # using pandas library (pd) to read the csv file.

df_test2 = removeConstant(df_test2, 1)
df_test2.shape

# 결측치를 처리한다.
test2 = handleMissingValue(df_test2)
test2.head(5)

# 셀 전압과 온도 데이터의 열의 인덱스 확인.
columns_test2 = test2.columns
for i in range(len(columns_test2)):
 print(i, columns_test2[i])

 # Collect voltage and temperature data
columns_vt = columns_test2[1:209]
print(columns_vt)
df_test2 = test2[columns_vt]
df_test2.head(5)

# Save the processed (train) data for later
df_test2.to_csv('./data/preprocessed/test/Test07_NG_dchg.csv', index = False)

0 Time
1 M01CV01
2 M01CV02
3 M01CV03
4 M01CV04
5 M01CV05
6 M01CV06
7 M01CV07
8 M01CV08
9 M01CV09
10 M01CV10
11 M01CV11
12 M02CV01
13 M02CV02
14 M02CV03
15 M02CV04
16 M02CV05
17 M02CV06
18 M02CV07
19 M02CV08
20 M02CV09
21 M02CV10
22 M02CV11
23 M03CV01
24 M03CV02
25 M03CV03
26 M03CV04
27 M03CV05
28 M03CV06
29 M03CV07
30 M03CV08
31 M03CV09
32 M03CV10
33 M03CV11
34 M04CV01
35 M04CV02
36 M04CV03
37 M04CV04
38 M04CV05
39 M04CV06
40 M04CV07
41 M04CV08
42 M04CV09
43 M04CV10
44 M04CV11
45 M05CV01
46 M05CV02
47 M05CV03
48 M05CV04
49 M05CV05
50 M05CV06
51 M05CV07
52 M05CV08
53 M05CV09
54 M05CV10
55 M05CV11
56 M06CV01
57 M06CV02
58 M06CV03
59 M06CV04
60 M06CV05
61 M06CV06
62 M06CV07
63 M06CV08
64 M06CV09
65 M06CV10
66 M06CV11
67 M07CV01
68 M07CV02
69 M07CV03
70 M07CV04
71 M07CV05
72 M07CV06
73 M07CV07
74 M07CV08
75 M07CV09
76 M07CV10
77 M07CV11
78 M08CV01
79 M08CV02
80 M08CV03
81 M08CV04
82 M08CV05
83 M08CV06
84 M08CV07
85 M08CV08
86 M08CV09
87 M08CV10
88 M08CV11
89 M09CV01
90 M09CV02
91 M09CV03
9

In [29]:
# PCA
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.decomposition import PCA
# plot utilities
import matplotlib.pyplot as plt
import plotly.graph_objects as go
# PCA 차원 축소 테스트
df_pca = pd.read_csv('./data/preprocessed/train/1000_chg.csv')
print("Initial data: ./data/preprocessed/train/1000_chg.csv")
print(df_pca.shape)
max_column = df_pca.shape[0]
max_row = df_pca.shape[1]
x = df_pca
pca = PCA(n_components=2)
principalComponents = pca.fit_transform(x)
principalDf = pd.DataFrame(data = principalComponents,
 columns = ['principal component 1', 'principal component 2'])
print("principalDf : ")
print(principalDf.head(5))
print("After PCA:")
pca_1 = np.array(principalDf['principal component 1'] )
pca_2 = np.array(principalDf['principal component 2'] )
# Create traces
fig = go.Figure()
fig.add_trace(go.Scatter(x = pca_1,
 y = pca_2,
 mode = 'lines+markers',
 name = 'PCA'))
fig.update_layout(title = 'PCA 분석 결과',
 xaxis_title = 'principal component 1',
 yaxis_title = 'principal component 2')
fig.show()

Initial data: ./data/preprocessed/train/1000_chg.csv
(6009, 208)
principalDf : 
   principal component 1  principal component 2
0              18.634349               0.564999
1              18.635771               0.571294
2              18.635892               0.570799
3              18.634366               0.564929
4              18.666694               0.565826
After PCA:
